# 🎙️ NLP Assignment 2: Low-Resource Speech-to-Text Translation
## Irish → English Cascaded Pipeline

**Module:** Natural Language Processing (MSc AI)  
**Deadline:** 10th May 2026  
**Weighting:** 50% of module

---

This notebook will guide you through building a cascaded **Automatic Speech Recognition (ASR) → Machine Translation (MT)** pipeline for translating Irish speech into English text.

### How to use this notebook
- Cells marked `# ✅ PROVIDED` are complete — run them as-is.
- Cells marked `# 📝 YOUR CODE HERE` require you to write your own code.
- Markdown cells with 💡 are hints. Read them before coding.
- Don't skip sections — each depends on the previous.

### Recommended runtime
Go to **Runtime → Change runtime type → T4 GPU** before starting.

---

### Table of Contents
1. [Setup & Installation](#setup)
2. [Dataset Loading & Exploration](#data)
3. [Audio Preprocessing](#audio)
4. [Baseline ASR — Whisper](#asr)
5. [Machine Translation — NLLB](#mt)
6. [Full Pipeline](#pipeline)
7. [Evaluation](#eval)
8. [Error Analysis](#error)
9. [Improved System](#improved)
10. [Final Results Table](#results)

---
## Section 1: Setup & Installation <a id='setup'></a>

In [1]:
# ✅ PROVIDED — Install all required libraries
# This may take 2-3 minutes on first run.

!pip install -q transformers datasets evaluate sacrebleu jiwer librosa soundfile torchaudio
!pip install -q accelerate>=0.26.0

print('✅ Libraries installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.7 MB/s eta 0:00:00a 0:00:01
✅ Libraries installed.


In [2]:
# ✅ PROVIDED — Imports

import os
import json
import time
import random
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch

from pathlib import Path
from IPython.display import Audio, display

from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoTokenizer, AutoModelForSeq2SeqLM,
    pipeline
)

import evaluate

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

🖥️  Device: cuda
   GPU: Tesla T4


---
## Section 2: Dataset Loading & Exploration <a id='data'></a>

For this assignment, evaluation should use the **IWSLT 2023 Irish-English development split**.

Why this split?
- The `iwslt2026_ga-eng` repository contains the blind 2026 test set, so it does not include English reference translations for BLEU/chrF++.
- Your instructor approved using `iwslt2023_ga-eng/dev`, which contains the references needed for evaluation.

**Dataset structure used here:**
```
iwslt2023_ga-eng/
├── dev/
│   ├── stamped.tsv          # metadata: audio path, start, end
│   ├── txt/
│   │   └── dev.eng         # English reference translations
│   └── wav/                # audio files
└── README.md
```

> ⚠️ In your report, document that evaluation was run on `iwslt2023_ga-eng/dev` from the linked submodule.

In [3]:
# ✅ PROVIDED — Clone the dataset

!git clone https://github.com/shashwatup9k/iwslt2023_ga-eng.git 2>/dev/null || echo 'Already cloned.'

# Inspect what was downloaded
print('\n📂 Repository contents:')
!find iwslt2023_ga-eng -type f | sort | head -40


📂 Repository contents:
iwslt2023_ga-eng/dev/stamped.tsv
iwslt2023_ga-eng/dev/txt/dev.eng
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_000.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_001.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_002.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_008.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_009.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_012.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_013.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_014.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_018.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_019.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_022.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_023.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_024.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_025.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_026.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_028.wav
iwslt2023_ga-e

In [4]:
# 📝 YOUR CODE HERE
# Task: Set the correct paths to the evaluation split based on the actual repo structure.

from pathlib import Path

DATA_ROOT = Path("iwslt2023_ga-eng")
DEV_ROOT = DATA_ROOT / "dev"

# Approved evaluation split: iwslt2023_ga-eng/dev
AUDIO_DIR = DEV_ROOT / "wav"
TRANS_FILE = DEV_ROOT / "stamped.tsv"   # metadata: audio path, start, end
REF_FILE = DEV_ROOT / "txt" / "dev.eng" # English reference translations

print('Audio dir exists:', AUDIO_DIR.exists())
print('Metadata file exists:', TRANS_FILE.exists())
print('References exist:', REF_FILE.exists())
print('Audio dir path:', AUDIO_DIR)
print('Metadata path:', TRANS_FILE)
print('Reference path:', REF_FILE)


Audio dir exists: True
Metadata file exists: True
References exist: True
Audio dir path: iwslt2023_ga-eng/dev/wav
Metadata path: iwslt2023_ga-eng/dev/stamped.tsv
Reference path: iwslt2023_ga-eng/dev/txt/dev.eng


In [5]:
# 📝 YOUR CODE HERE
# Task: Load the metadata and references into Python lists.
# Print the first examples so you can verify they line up.

# HINT: Each line in the file corresponds to one utterance.
# Use open() and .strip() to clean whitespace.

with open(TRANS_FILE, encoding="utf-8") as f:
    audio_ids = [line.strip().split('\t')[0] for line in f if line.strip()]

with open(REF_FILE, encoding="utf-8") as f:
    english_refs = [line.strip() for line in f if line.strip()]

# TODO: Print dataset statistics
# - How many utterances are there?
# - Do the counts match?
# - What does the first example look like?

print(f'Number of audio items: {len(audio_ids)}')
print(f'Number of references:  {len(english_refs)}')
print(f'Counts match:          {len(audio_ids) == len(english_refs)}')
print()

for i in range(3):
    print(f'--- Example {i+1} ---')
    print(f'Audio item: {audio_ids[i]}')
    print(f'English:    {english_refs[i]}')
    print()

Number of audio items: 1120
Number of references:  1120
Counts match:          True

--- Example 1 ---
Audio item: wav/iwslt2023_ga-eng_z0001_000.wav
English:    I am indeed, he answered.

--- Example 2 ---
Audio item: wav/iwslt2023_ga-eng_z0001_001.wav
English:    In a report, written by inspectors from the Department

--- Example 3 ---
Audio item: wav/iwslt2023_ga-eng_z0001_002.wav
English:    Michael D. Higgins has always been independent.



In [6]:
# ✅ PROVIDED — Get list of audio files

audio_files = sorted(list(AUDIO_DIR.glob('*.wav')))
print(f'Found {len(audio_files)} audio files.')
if audio_files:
    print('First 5:', [f.name for f in audio_files[:5]])

Found 1120 audio files.
First 5: ['iwslt2023_ga-eng_z0001_000.wav', 'iwslt2023_ga-eng_z0001_001.wav', 'iwslt2023_ga-eng_z0001_002.wav', 'iwslt2023_ga-eng_z0001_008.wav', 'iwslt2023_ga-eng_z0001_009.wav']


In [7]:
print("AUDIO_DIR =", AUDIO_DIR)
print("Exists =", AUDIO_DIR.exists())
!find iwslt2023_ga-eng/dev/wav -name "*.wav" | wc -l
!find iwslt2023_ga-eng/dev/wav -name "*.wav" | head
!find iwslt2023_ga-eng/dev/wav -name "*.wav" | tail


AUDIO_DIR = iwslt2023_ga-eng/dev/wav
Exists = True
1120
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_539.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_051.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_858.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0002_069.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_608.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_481.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_398.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_040.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_904.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_459.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0002_192.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0002_016.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0002_079.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_751.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0002_111.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0002_470.wav
iwslt2023_ga-eng/dev/wav/iwslt2023_ga-eng_z0001_

---
## Section 3: Audio Inspection & Preprocessing <a id='audio'></a>

Before feeding audio to an ASR model, you need to understand its properties:
- **Sample rate** — Whisper expects 16,000 Hz (16 kHz)
- **Duration** — very short or very long clips may cause issues
- **Format** — most models expect mono-channel float arrays

💡 **Tip:** Librosa's `load()` function can resample audio automatically.

In [8]:
# ✅ PROVIDED — Inspect one audio file

if audio_files:
    sample_file = audio_files[0]
    waveform, sr = librosa.load(sample_file, sr=None)  # sr=None preserves original rate

    print(f'File:        {sample_file.name}')
    print(f'Sample rate: {sr} Hz')
    print(f'Duration:    {len(waveform)/sr:.2f} seconds')
    print(f'Shape:       {waveform.shape}')
    print(f'Min/Max:     {waveform.min():.3f} / {waveform.max():.3f}')

    # Listen to it!
    display(Audio(waveform, rate=sr))

File:        iwslt2023_ga-eng_z0001_000.wav
Sample rate: 48000 Hz
Duration:    1.86 seconds
Shape:       (89460,)
Min/Max:     -0.672 / 0.345


In [9]:
# 📝 YOUR CODE HERE
# Task: Write a function to load and preprocess an audio file.
# Requirements:
#   - Resample to 16000 Hz (Whisper's required rate)
#   - Return a float32 numpy array
#   - Handle mono audio only

TARGET_SR = 16000

def load_audio(filepath, target_sr=TARGET_SR):
    """
    Load an audio file and resample to target_sr.
    Returns: numpy array (float32), sample rate
    """
    waveform, sr = librosa.load(filepath, sr=target_sr, mono=True)
    waveform = waveform.astype(np.float32)
    return waveform, sr

# Test your function
if audio_files:
    wav, sr = load_audio(audio_files[0])
    assert sr == TARGET_SR, f'Sample rate should be {TARGET_SR}, got {sr}'
    assert wav.dtype == np.float32, 'Array should be float32'
    print(f'✅ load_audio works. Duration: {len(wav)/sr:.2f}s')

✅ load_audio works. Duration: 1.86s


In [10]:
# 📝 YOUR CODE HERE
# Task: Compute and report basic statistics for the audio dataset.
# Report:
#   - Total number of audio files
#   - Average, min, max duration in seconds
#   - Total hours of audio

# Compute and report basic statistics for the audio dataset

#sample_files = audio_files[:50]  # Use subset if dataset is large
durations = []

for f in audio_files:
    wav, sr = load_audio(f)
    duration_sec = len(wav) / sr
    durations.append(duration_sec)

print('=== Audio Statistics ===')
print(f'audio files size:      {len(audio_files)}')
print(f'Average duration:   {np.mean(durations):.2f} s')
print(f'Min duration:       {np.min(durations):.2f} s')
print(f'Max duration:       {np.max(durations):.2f} s')
print(f'Total audio time:   {np.sum(durations):.2f} s')
print(f'Total audio hours:  {np.sum(durations) / 3600:.4f} h')


=== Audio Statistics ===
audio files size:      1120
Average duration:   3.32 s
Min duration:       1.19 s
Max duration:       7.99 s
Total audio time:   3714.90 s
Total audio hours:  1.0319 h


---
## Section 4: Baseline ASR — Whisper <a id='asr'></a>

**Whisper** (Radford et al., 2022) is a multilingual ASR model from OpenAI, trained on 680,000 hours of multilingual audio. It supports Irish and requires 16 kHz mono audio input.

### Available Whisper model sizes:

| Model | Parameters | Recommended for |
|-------|-----------|----------------|
| `whisper-tiny` | 39M | Quick testing |
| `whisper-base` | 74M | Baseline (fast) |
| `whisper-small` | 244M | Better quality |
| `whisper-medium` | 769M | Good baseline |
| `whisper-large-v3` | 1.5B | Strongest (slow on CPU) |

💡 **Recommendation for baseline:** Start with `openai/whisper-small` for a reasonable speed/quality tradeoff on Colab.

In [11]:
# 📝 YOUR CODE HERE
# Task: Load a Whisper model and processor.
#
# Requirements:
#   - Choose a Whisper model appropriate for your baseline
#   - Load the processor and model
#   - Move model to DEVICE
#   - Justify your choice in a comment

# 📝 YOUR CODE HERE
# Task: Load a Whisper model and processor.
#
# Source adapted from the Hugging Face Whisper model page:
# https://huggingface.co/openai/whisper-small

ASR_MODEL_ID = "openai/whisper-small"

print(f'Loading ASR model: {ASR_MODEL_ID}...')

whisper_processor = WhisperProcessor.from_pretrained(ASR_MODEL_ID)
whisper_model = WhisperForConditionalGeneration.from_pretrained(ASR_MODEL_ID)

whisper_model = whisper_model.to(DEVICE)
whisper_model.eval()

# Clean generation setup
whisper_model.generation_config.forced_decoder_ids = None
whisper_model.generation_config.task = "transcribe"
whisper_model.generation_config.language = None

print(f'✅ Loaded {ASR_MODEL_ID}')
print(f'   Parameters: {sum(p.numel() for p in whisper_model.parameters()):,}')



Loading ASR model: openai/whisper-small...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

✅ Loaded openai/whisper-small
   Parameters: 241,734,912


In [12]:
# 📝 YOUR CODE HERE
# Task: Write a function to transcribe a single audio file using Whisper.
#
# Steps:
#   1. Load and preprocess audio (use your load_audio function)
#   2. Create input features with the Whisper processor
#   3. Generate transcription tokens (force Irish language: forced_decoder_ids)
#   4. Decode the output tokens to text
#   5. Return the transcription string

# 📝 YOUR CODE HERE
# Task: Write a function to transcribe a single audio file using Whisper.
#
# Source adapted from the Hugging Face Whisper model page:
# https://huggingface.co/openai/whisper-small

# Source adapted from the Hugging Face Whisper model page:
# https://huggingface.co/openai/whisper-small

def transcribe_with_whisper(audio_path, processor, model, language=None):
    """
    Transcribe a single audio file using Whisper.
    """
    waveform, sr = load_audio(audio_path)

    inputs = processor(
        waveform,
        sampling_rate=sr,
        return_tensors='pt'
    )

    input_features = inputs.input_features.to(model.device)

    if language is not None:
        forced_ids = processor.get_decoder_prompt_ids(
            language=language,
            task='transcribe'
        )
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=forced_ids
        )
    else:
        predicted_ids = model.generate(input_features)

    transcription = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    return transcription

# Test on one file
if audio_files:
    result = transcribe_with_whisper(audio_files[0], whisper_processor, whisper_model, language=None)
    print(f'Audio file:         {audio_files[0].name}')
    print(f'Test transcription: {result}')



The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

Audio file:         iwslt2023_ga-eng_z0001_000.wav
Test transcription:  Tom Gadaian, Adagrias.


In [13]:
# 📝 YOUR CODE HERE
# Task: Run ASR on a subset of the development/test set.
#
# - Process the first MAX_SAMPLES audio files (start with 50 for speed)
# - Store transcriptions in a list
# - Track and print average processing time per file
#
# 💡 TIP: Use tqdm for a progress bar:
#   from tqdm import tqdm

from tqdm import tqdm

MAX_SAMPLES = 50  # Increase later when the pipeline is stable

asr_hypotheses = []
asr_times = []

for audio_path in tqdm(audio_files[:MAX_SAMPLES], desc="Running ASR"):
    start_time = time.time()

    try:
        transcription = transcribe_with_whisper(audio_path, whisper_processor, whisper_model)
    except Exception as e:
        print(f"Error on {audio_path.name}: {e}")
        transcription = ""

    elapsed = time.time() - start_time

    asr_hypotheses.append(transcription)
    asr_times.append(elapsed)

print(f'Total ASR hypotheses: {len(asr_hypotheses)}')
print(f'Average time per file: {np.mean(asr_times):.2f}s')


Running ASR: 100%|██████████| 50/50 [00:21<00:00,  2.32it/s]

Total ASR hypotheses: 50
Average time per file: 0.43s


---
## Section 5: Machine Translation — NLLB <a id='mt'></a>

**NLLB-200** (No Language Left Behind, Meta AI, 2022) is a multilingual MT model covering 200 languages, including Irish (`gle_Latn`).

### Available NLLB variants:

| Model | Size | Notes |
|-------|------|-------|
| `facebook/nllb-200-distilled-600M` | 600M | Recommended for baseline |
| `facebook/nllb-200-distilled-1.3B` | 1.3B | Better quality, slower |
| `facebook/nllb-200-1.3B` | 1.3B | Non-distilled |

💡 **Language codes:** Irish = `gle_Latn`, English = `eng_Latn`

In [14]:
# 📝 YOUR CODE HERE
# Task: Load the NLLB-200 model and tokeniser.

MT_MODEL_ID = 'facebook/nllb-200-distilled-600M'

SRC_LANG = 'gle_Latn'  # Irish
TGT_LANG = 'eng_Latn'  # English

print(f'Loading MT model: {MT_MODEL_ID}...')

mt_tokenizer = AutoTokenizer.from_pretrained(MT_MODEL_ID)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL_ID)

mt_model = mt_model.to(DEVICE)
mt_model.eval()

print(f'✅ Loaded {MT_MODEL_ID}')
print(f'   Parameters: {sum(p.numel() for p in mt_model.parameters()):,}')

Loading MT model: facebook/nllb-200-distilled-600M...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ Loaded facebook/nllb-200-distilled-600M
   Parameters: 1,402,138,624


In [15]:
# 📝 YOUR CODE HERE
# Task: Write a function to translate Irish text to English using NLLB.
#
# Source adapted from Hugging Face NLLB usage:
# https://huggingface.co/facebook/nllb-200-distilled-600M

def translate_with_nllb(text, tokenizer, model, src_lang=SRC_LANG, tgt_lang=TGT_LANG, max_new_tokens=256):
    """
    Translate Irish text to English.
    Returns: translated string
    """
    if not text or not text.strip():
        return ''

    tokenizer.src_lang = src_lang

    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_new_tokens=max_new_tokens
    )

    translation = tokenizer.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0]

    return translation

# Test it!
test_irish = 'Dia duit, conas atá tú?'
translation = translate_with_nllb(test_irish, mt_tokenizer, mt_model)
print(f'Irish:   {test_irish}')
print(f'English: {translation}')


Irish:   Dia duit, conas atá tú?
English: Dia duit, how are you?


---
## Section 6: Full Cascaded Pipeline <a id='pipeline'></a>

Now combine ASR → MT into one end-to-end function.

```
Audio file
    │
    ▼ load_audio()
Waveform (16kHz)
    │
    ▼ transcribe_with_whisper()
Irish transcript (text)
    │
    ▼ translate_with_nllb()
English translation (text)
```

In [17]:
# 📝 YOUR CODE HERE
# Task: Write the full pipeline function.

def speech_to_text_translate(audio_path,
                             asr_processor, asr_model,
                             mt_tokenizer, mt_model):
    """
    Full cascaded speech translation: audio file -> English text.
    Returns: (irish_transcript, english_translation)
    """
    irish_transcript = transcribe_with_whisper(
        audio_path,
        asr_processor,
        asr_model
    )

    english_translation = translate_with_nllb(
        irish_transcript,
        mt_tokenizer,
        mt_model
    )

    return irish_transcript, english_translation

# Test on a sample
if audio_files:
    transcript, translation = speech_to_text_translate(
        audio_files[0], whisper_processor, whisper_model, mt_tokenizer, mt_model
    )
    print(f'ASR Transcript: {transcript}')
    print(f'MT Translation: {translation}')
    print(f'Reference:      {english_refs[0]}')


ASR Transcript:  Tom Gadaian, Adagrias.
MT Translation: Tom Gadaian, Adagrias.
Reference:      I am indeed, he answered.


In [18]:
# 📝 YOUR CODE HERE
# Task: Run the full pipeline on MAX_SAMPLES files.
# Save: transcripts, translations, and any failed indices.

baseline_transcripts = []
baseline_translations = []
failed_indices = []

for i, audio_path in enumerate(tqdm(audio_files[:MAX_SAMPLES], desc="Running full pipeline")):
    try:
        transcript, translation = speech_to_text_translate(
            audio_path,
            whisper_processor,
            whisper_model,
            mt_tokenizer,
            mt_model
        )
    except Exception as e:
        print(f"Error on index {i} ({audio_path.name}): {e}")
        transcript, translation = "", ""
        failed_indices.append(i)

    baseline_transcripts.append(transcript)
    baseline_translations.append(translation)

print(f'Processed: {len(baseline_translations)} files')
print(f'Failed: {len(failed_indices)} files')


Running full pipeline: 100%|██████████| 50/50 [00:35<00:00,  1.42it/s]

Processed: 50 files
Failed: 0 files


In [19]:
# ✅ PROVIDED — Save baseline outputs

os.makedirs('results', exist_ok=True)

with open('results/baseline_hypotheses.txt', 'w') as f:
    for line in baseline_translations:
        f.write(line + '\n')

with open('results/references.txt', 'w') as f:
    for line in english_refs[:MAX_SAMPLES]:
        f.write(line + '\n')

print('✅ Saved results/baseline_hypotheses.txt')
print('✅ Saved results/references.txt')

✅ Saved results/baseline_hypotheses.txt
✅ Saved results/references.txt


---
## Section 7: Evaluation <a id='eval'></a>

We evaluate using two standard metrics:

| Metric | Description | Why use it? |
|--------|-------------|-------------|
| **BLEU** | N-gram overlap between hypothesis and reference | Standard MT metric |
| **chrF++** | Character + word n-gram F-score | More robust for morphologically rich languages |

💡 Always use **SacreBLEU** for reproducible BLEU scores — never implement BLEU from scratch.

In [20]:
# ✅ PROVIDED — Evaluation helper functions

import sacrebleu

def compute_bleu(hypotheses, references):
    """
    Compute corpus BLEU using SacreBLEU.
    Args:
        hypotheses: list of predicted strings
        references: list of reference strings
    Returns: BLEU score (float)
    """
    # SacreBLEU expects references wrapped in a list of lists
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    return bleu.score

def compute_chrf(hypotheses, references):
    """
    Compute chrF++ using SacreBLEU.
    """
    chrf = sacrebleu.corpus_chrf(hypotheses, [references], word_order=2)
    return chrf.score

def compute_coverage(hypotheses):
    """Proportion of non-empty outputs."""
    n_empty = sum(1 for h in hypotheses if not h.strip())
    return 1.0 - (n_empty / max(len(hypotheses), 1))

def compute_repetition_rate(hypotheses, threshold=0.3):
    """Proportion of outputs where the same word appears too frequently."""
    flagged = 0
    for h in hypotheses:
        words = h.split()
        if len(words) < 4:
            continue
        from collections import Counter
        freqs = Counter(words)
        if freqs.most_common(1)[0][1] / len(words) > threshold:
            flagged += 1
    return flagged / max(len(hypotheses), 1)

print('✅ Evaluation functions loaded.')

✅ Evaluation functions loaded.


In [21]:
# 📝 YOUR CODE HERE
# Task: Evaluate your baseline system.
# - Filter out any failed samples (where hypothesis or reference is empty)
# - Compute BLEU, chrF++, coverage, repetition rate
# - Print a summary table

# Evaluate your baseline system

refs_subset = english_refs[:MAX_SAMPLES]

valid_hyps = []
valid_refs = []

for hyp, ref in zip(baseline_translations, refs_subset):
    if hyp.strip() and ref.strip():
        valid_hyps.append(hyp)
        valid_refs.append(ref)

bleu = compute_bleu(valid_hyps, valid_refs)
chrf = compute_chrf(valid_hyps, valid_refs)
coverage = compute_coverage(baseline_translations)
rep_rate = compute_repetition_rate(baseline_translations)

print('=== Baseline System Results ===')
print(f'Samples evaluated:   {len(valid_hyps)}')
print(f'BLEU:                {bleu:.2f}')
print(f'chrF++:              {chrf:.2f}')
print(f'Coverage:            {coverage:.1%}')
print(f'Repetition rate:     {rep_rate:.1%}')


=== Baseline System Results ===
Samples evaluated:   50
BLEU:                0.65
chrF++:              13.25
Coverage:            100.0%
Repetition rate:     0.0%


---
## Section 8: Error Analysis <a id='error'></a>

Quantitative metrics tell you *how much* a system gets wrong — error analysis tells you *why*.

### Common error types to look for:

| Error Type | Description | Example |
|-----------|-------------|--------|
| **ASR substitution** | Wrong word transcribed | "sráid" → "sraith" |
| **ASR deletion** | Missing words | "tá sé ann" → "sé ann" |
| **ASR hallucination** | Model confabulates plausible-sounding but wrong text | |
| **MT untranslated** | Irish word passed through to English output | |
| **MT mistranslation** | Wrong meaning | "leabhar" → "book" ✅ vs "leabhar" → "library" ❌ |
| **MT over-generation** | Unexplained extra text in output | |
| **Error propagation** | ASR error causes MT error | |

💡 Aim for a systematic analysis of at least **20 examples**.

In [23]:
# ✅ PROVIDED — Side-by-side comparison helper

# ✅ PROVIDED — Side-by-side comparison helper

def show_examples(n=10, start=0):
    """Display examples in a readable format for error analysis."""
    for i in range(start, min(start + n, len(baseline_translations))):
        print(f'--- Example {i+1} ---')
        print(f'Audio file:             {audio_files[i].name}')
        print(f'Irish transcript (ASR): {baseline_transcripts[i]}')
        print('Irish reference:        unavailable in this dataset copy')
        print(f'English translation:    {baseline_translations[i]}')
        print(f'English reference:      {english_refs[i]}')
        print()

show_examples(n=5)


--- Example 1 ---
Audio file:             iwslt2023_ga-eng_z0001_000.wav
Irish transcript (ASR):  Tom Gadaian, Adagrias.
Irish reference:        unavailable in this dataset copy
English translation:    Tom Gadaian, Adagrias.
English reference:      I am indeed, he answered.

--- Example 2 ---
Audio file:             iwslt2023_ga-eng_z0001_001.wav
Irish transcript (ASR):  Idoris, a screve kickery, own Rhine.
Irish reference:        unavailable in this dataset copy
English translation:    Idoris, a screve kickery, owns the Rhine.
English reference:      In a report, written by inspectors from the Department

--- Example 3 ---
Audio file:             iwslt2023_ga-eng_z0001_002.wav
Irish transcript (ASR):  The Michael D Higgins Reeve Navs Block
Irish reference:        unavailable in this dataset copy
English translation:    The Michael D Higgins Reeve Navs Block is a film about the life of a young boy named Michael D Higgins.
English reference:      Michael D. Higgins has always been indep

In [24]:
# 📝 YOUR CODE HERE
# Task: Manually annotate 20+ examples with error categories.
#
# Create a list of dictionaries, one per example, containing:
#   - 'id': example index
#   - 'asr_errors': list of error types in ASR output
#   - 'mt_errors': list of error types in MT output
#   - 'notes': your comment on the example

# Example annotation:
error_annotations = [
    {
        'id': 0,
        'asr_errors': ['substitution'],
        'mt_errors': ['mistranslation'],
        'notes': 'ASR appears to contain lexical errors, which likely affected the English translation.'
    },
    {
        'id': 1,
        'asr_errors': ['deletion'],
        'mt_errors': ['under-translation'],
        'notes': 'Part of the source content seems to be missing in the ASR output, and the MT output is incomplete.'
    },
    {
        'id': 2,
        'asr_errors': ['hallucination'],
        'mt_errors': ['mistranslation'],
        'notes': 'The ASR output is not reliable, so the translation drifts from the reference meaning.'
    },
    # Add more examples here after reviewing real outputs
]

from collections import Counter

all_asr_errors = [e for ann in error_annotations for e in ann['asr_errors']]
all_mt_errors = [e for ann in error_annotations for e in ann['mt_errors']]

print('ASR error types:', Counter(all_asr_errors))
print('MT error types: ', Counter(all_mt_errors))
print(f'Total annotated examples: {len(error_annotations)}')

ASR error types: Counter({'substitution': 1, 'deletion': 1, 'hallucination': 1})
MT error types:  Counter({'mistranslation': 2, 'under-translation': 1})
Total annotated examples: 3


---
## Section 9: Improved System <a id='improved'></a>

You must implement **at least one meaningful improvement** over your baseline.

### Possible directions (choose one or more):

| Direction | Description | Effort |
|-----------|-------------|--------|
| 🔄 **Different ASR model** | Try `whisper-medium` or a Wav2Vec2 Irish model | Low |
| 🔄 **Different MT model** | Try `nllb-1.3B` or `Helsinki-NLP/opus-mt-ga-en` | Low |
| 🔧 **ASR language forcing** | Ensure Whisper transcribes in Irish, not English | Low |
| 🔧 **Beam search tuning** | Increase `num_beams` for MT | Low |
| ✂️ **Output filtering** | Remove empty/degenerate outputs before MT | Medium |
| 🧹 **Text normalisation** | Clean ASR output before MT (punctuation, case) | Medium |
| 🎛️ **Audio augmentation** | Normalise audio volume before ASR | Medium |

You must **justify your choice** in your report and **compare results** against the baseline.

In [26]:
# 📝 YOUR CODE HERE
# Run benchmark experiments from the repo script

!git clone https://github.com/simon-gobin/LLM_cascade_translation_benchmark.git
%cd LLM_cascade_translation_benchmark
!ls



Cloning into 'LLM_cascade_translation_benchmark'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 10 (delta 0), reused 10 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), 364.24 KiB | 13.01 MiB/s, done.
/content/LLM_cascade_translation_benchmark
benchmark.py			 experiment_tracking_notes.pdf
benchmark_sources.md		 experiment_tracking_notes.tex
experiment_notes_for_report.md	 README.md
experiment_notes_for_report.pdf  Student_Colab_Notebook_in_progress.ipynb


In [27]:
!python benchmark.py \
  --dataset-root iwslt2023_ga-eng \
  --experiments baseline \
  --max-samples 10



Traceback (most recent call last):
  File "/content/LLM_cascade_translation_benchmark/benchmark.py", line 608, in <module>
    main()
  File "/content/LLM_cascade_translation_benchmark/benchmark.py", line 556, in main
    samples = load_dev_samples(args.dataset_root, args.start_index, args.max_samples)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/LLM_cascade_translation_benchmark/benchmark.py", line 175, in load_dev_samples
    raise FileNotFoundError(f"Missing wav dir: {wav_dir}")
FileNotFoundError: Missing wav dir: iwslt2023_ga-eng/dev/wav


---
## Section 10: Final Results Table <a id='results'></a>

Your report must include a results table. Build it here.

In [ ]:
# 📝 YOUR CODE HERE
# Task: Build and display a results table comparing baseline and improved system.

results = pd.DataFrame([
    {
        'System': f'Baseline ({ASR_MODEL_ID} + {MT_MODEL_ID})',
        'BLEU': bleu,
        'chrF++': chrf,
        'Coverage': f'{coverage:.1%}',
        'Rep. Rate': f'{rep_rate:.1%}',
    },
    {
        'System': 'Improved (describe your system)',  # TODO: Update description
        'BLEU': 0.0,    # TODO: Replace with real values
        'chrF++': 0.0,  # TODO: Replace with real values
        'Coverage': 'N/A',
        'Rep. Rate': 'N/A',
    }
])

results = results.set_index('System')
print('=== Final Results ===')
display(results)

# Save to file
results.to_csv('results/final_results.csv')
print('\nSaved to results/final_results.csv')

In [ ]:
# 📝 YOUR CODE HERE
# Task: Write a brief (3-5 sentence) interpretation of your results in this cell.
# This will help you draft the Results section of your report.

# TODO: Fill in your analysis
print("""
=== Results Interpretation ===

[Write 3-5 sentences here interpreting your results. For example:]
- How do BLEU/chrF++ compare between systems?
- Which errors are most common and why?
- Was the improvement meaningful? What might explain it?
- What are the key limitations of your approach?
""")

---
## ✅ Submission Checklist

Before submitting, confirm:

- [ ] All cells run top-to-bottom without errors
- [ ] `results/baseline_hypotheses.txt` saved
- [ ] `results/final_results.csv` saved
- [ ] Error analysis covers at least 20 examples
- [ ] Improved system is distinct from baseline and justified
- [ ] README.md explains how to run your notebook
- [ ] ACL-format report is complete (6–8 pages)
- [ ] All AI tool usage is declared in the report appendix
- [ ] All random seeds are set and documented

**Good luck! 🍀**